# Therapeptides

In [2]:
import os
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv('1_/therapeptides_clean.csv')
df

,Function,Sequence
0,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS
1,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",AAAAAAAAAK
2,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",AAAAAAAAAR
3,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",AAAAAAAIKMLMDLVNERIMALNKKAKK
4,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",AAAAAAK
...,...,...
54489,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",YCYCRRRFCVCVGR
54490,Antimicrobial|Antibacterial|Anti-gram-,YKKALKKLAKL
54491,Antimicrobial|Antibacterial|Anti-gram-,YKKALKKLAKLL
54492,"Anticancer,Antimicrobial",YKQCHKKGGKKGSG


In [4]:
df_mftp_train = pd.read_csv('mftp_train.csv')
df_mftp_test = pd.read_csv('mftp_test.csv')

df_mftp_all = pd.concat([df_mftp_train, df_mftp_test]).reset_index(drop=True)


In [5]:
df_mftp_all

,sequence,label
0,ARRRRCSDRFRNCPADEALCGRRRR,000000000000000010000
1,RLHHRLHRRLHRLHRRLHRLHHRLHRRLH,000000000000000010000
2,RLFGAEVGWLALHHN,000000000100000000000
3,IIGPVLGMVGSALGGLLKKI,010000000000000000000
4,EKDERF,000000001000000000000
...,...,...
9869,SLEEEIRFL,000010000000000000000
9870,SIITMTKEAKLPQSWKQIACRLYNTC,010000000000000000000
9871,GNFRYLAPP,001000000000000000000
9872,FFGHLFKLATKIIPSFFRRKNQ,010000000000000000000


In [6]:
df_pretrain = df[~df["Sequence"].str.upper().str.strip().isin(
    df_mftp_all["sequence"].str.upper().str.strip()
)]

df_pretrain = df_pretrain.reset_index(drop=True)
df_pretrain

,Function,Sequence
0,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS
1,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",AAAAAAAAAK
2,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",AAAAAAAAAR
3,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",AAAAAAAIKMLMDLVNERIMALNKKAKK
4,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",AAAAAAK
...,...,...
47011,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",YWFHWK
47012,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",YCYCRRRFCVCVGR
47013,Antimicrobial|Antibacterial|Anti-gram-,YKKALKKLAKL
47014,Antimicrobial|Antibacterial|Anti-gram-,YKKALKKLAKLL


In [8]:
df_pretrain['Function'].value_counts()

Function
Antimicrobial                                                                                                                                                  9679
Neuropeptide                                                                                                                                                   6103
Antimicrobial|Antibacterial|Anti-gram+,Antimicrobial|Antibacterial|Anti-gram-                                                                                  5682
Antimicrobial|Antibacterial                                                                                                                                    3792
Antimicrobial|Antibacterial|Anti-gram+,Antimicrobial|Antibacterial|Anti-gram-,Antimicrobial|Antifungal                                                         2501
                                                                                                                                                               ... 
Antimic

In [9]:
df_pretrain.to_csv('pretrain.csv', index=False)

In [12]:

with open("2_mmseq/pretrain.fasta", "w") as f:
    for i, s in enumerate(df_pretrain["Sequence"]):
        f.write(f">seq{i}\n{s}\n")

## After Cluster

In [14]:
rep = []
with open("2_mmseq/pretrain_c90_rep_seq.fasta") as f:
    for line in f:
        if not line.startswith(">"):
            rep.append(line.strip())

rep_set = set(rep)
df_pretrain_mmseqs = df_pretrain[df_pretrain["Sequence"].isin(rep_set)].copy()

print("before:", len(df_pretrain))
print("after :", len(df_pretrain_mmseqs))

before: 47016
after : 32854


In [16]:
df_pretrain_mmseqs.to_csv('2_mmseq/pretrain_after_mmseqs.csv', index=False)

In [18]:
df_pretrain_mmseqs["Sequence"].to_csv('train.txt', index=False, header=False)

In [19]:
df_pretrain_mmseqs

,Function,Sequence
0,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS
1,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",AAAAAAAAAK
3,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",AAAAAAAIKMLMDLVNERIMALNKKAKK
4,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",AAAAAAK
5,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",AAAAAAPKKPAAAAAA
...,...,...
47010,Antimicrobial|Antibacterial,YPVKYPKL
47011,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",YWFHWK
47013,Antimicrobial|Antibacterial|Anti-gram-,YKKALKKLAKL
47014,Antimicrobial|Antibacterial|Anti-gram-,YKKALKKLAKLL


In [20]:
with open("pretrain.fasta", "w") as f:
    for i, s in enumerate(df_pretrain_mmseqs["Sequence"]):
        f.write(f">{i}\n{s}\n")

# Split train and val

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df_all = pd.read_csv("pretrain.csv")

train_df, val_df = train_test_split(
    df_all,
    test_size=0.05,
    random_state=42,
    shuffle=True
)

print("train:", len(train_df))
print("val:", len(val_df))

train: 31211
val: 1643


In [2]:
train_df

,Function,Sequence
18561,Antioxidant,MLELDPLNNLPRM
5172,Neuropeptide,ETTFTPRL
28934,Antimicrobial,VLGTILKLLKSL
22274,Antimicrobial|Antibacterial,RGAAAVRR
725,Antimicrobial|Antiparasitic,AGRQALKKYLLEELRKRGKKAFIAW
...,...,...
16850,"Antimicrobial|Antibacterial|Anti-gram+,Antimic...",LKRLYKRLAKLIKRLYRYLKKPVR
6265,Antimicrobial,FLISIPYSASIGGTATLTGTA
11284,"Anticancer,Antimicrobial",HHHHHHHHRRRRRRRR
860,"Anticancer,Antimicrobial,Drug_delivery|Cell_pe...",AIIYRDLIS


In [3]:
with open("pretrain_train.txt", "w") as f:
    for i, s in enumerate(train_df["Sequence"]):
        f.write(f">{i}\n{s}\n")
with open("pretrain_val.txt", "w") as f:
    for i, s in enumerate(val_df["Sequence"]):
        f.write(f">{i}\n{s}\n")